# 01 NumPy 简介与 ndarray 基础

> ### 本章目标
>
> - 理解 NumPy 是什么，以及它为什么是 Python 数据分析 / 科学计算生态的基石
> - 理解 ndarray 为什么比 Python list 快（连续内存 + C 实现 + 向量化）
> - 掌握 ndarray 的核心属性：`shape` / `ndim` / `size` / `dtype` / `itemsize` / `nbytes`
> - 掌握创建数组的常用方式：`np.array` / `arange` / `linspace` / `zeros` / `ones` / `full` / `empty` / `eye` / `identity` / `diag`
> - 掌握 dtype 类型体系、`astype` 转换、类型提升规则与整数溢出陷阱
> - 掌握数组与 Python 容器（list / tuple）的相互转换
>
> 📌 建议：边看边把代码 cell 逐个运行一遍，效果最佳。

---

In [1]:
import numpy as np

print(np.__version__)

2.4.4


## 1.1 NumPy 是什么？为什么需要它？

**一句话定义**：NumPy（Numerical Python）是 Python 做数值计算的基础库，它提供了核心数据结构 **ndarray**（N 维数组，N-dimensional array）以及基于它的一整套高效数学函数。

**它解决什么问题？**

- Python 内置的 `list` 是「万能容器」，可以装任意类型，但做数值运算慢得离谱；
- 科学计算需要「矩阵式」的批量运算：逐元素相加、矩阵乘法、统计汇总……
- 用纯 Python 写 `for` 循环做这些事，代码又长又慢；NumPy 让你**一行代码**完成同样的计算。

**为什么说它是生态基石？**

几乎所有的数据科学 / 科学计算库都构建在 NumPy 之上：

| 库 | 作用 | 与 NumPy 的关系 |
|---|---|---|
| pandas | 数据分析、表格处理 | DataFrame 底层就是 ndarray |
| scikit-learn | 机器学习 | 特征矩阵就是 ndarray |
| SciPy | 科学计算 | 大量函数直接操作 ndarray |
| Matplotlib | 绘图 | 直接接收 ndarray 作为数据 |

> 💡 **提示**：学好 NumPy，等于给后面学 pandas、机器学习铺好了地基。它是「数据科学第一课」。

---

### NumPy vs 纯 Python：直观对比

| 维度 | Python list | NumPy ndarray |
|---|---|---|
| 元素类型 | 可混装任意类型 | **同构**（统一 dtype） |
| 内存 | 指针数组 + 分散对象 | **连续内存块** |
| 运算方式 | 需手写循环 | **向量化**，一行搞定 |
| 速度 | 慢 | 快几十倍甚至上百倍 |
| 常用方法 | `sum()`、`append()` 等 | `arr.sum()`、`arr.mean()` 等 |

## 1.2 为什么 NumPy 快？——ndarray 的内存布局

NumPy 快主要靠三件事：

1. **同构定长**：所有元素类型相同、占用字节数相同；
2. **连续内存**：元素紧挨着存储在连续的内存块中，CPU 缓存命中率高；
3. **C 语言实现 + 向量化**：`arr.sum()` 等操作在底层用编译好的 C 代码批量执行，还能利用 SIMD 指令集。

而 Python `list` 里存的是**指向对象的指针**，每个对象分散在内存各处：

```mermaid
graph LR
    subgraph Python列表
        P["指针数组"] --> O0["对象0 任意类型"]
        P --> O1["对象1 任意类型"]
        P --> O2["对象2 任意类型"]
    end
    subgraph NumPy数组
        C["连续内存块"] --- E0["元素0"]
        C --- E1["元素1"]
        C --- E2["元素2"]
    end
```

> 💡 **提示**：list 每次取元素都要「指针跳转」，还要做类型检查；ndarray 是「顺着内存挨个读」，所以批量计算快得多。

下面实测一下求和性能。

In [2]:
import time

n = 1_000_000

# Python list：纯 Python 循环求和（逐一相加）
lst = list(range(n))
t0 = time.perf_counter()
s1 = sum(lst)
t1 = time.perf_counter()

# NumPy ndarray：向量化求和（底层是 C 实现）
arr = np.arange(n)
t2 = time.perf_counter()
s2 = arr.sum()
t3 = time.perf_counter()

print(f'Python sum(list) 耗时: {(t1 - t0) * 1000:.3f} ms, 结果 = {s1}')
print(f'NumPy  arr.sum() 耗时: {(t3 - t2) * 1000:.3f} ms, 结果 = {s2}')
print(f'加速比: {(t1 - t0) / (t3 - t2):.1f} 倍')

Python sum(list) 耗时: 13.032 ms, 结果 = 499999500000
NumPy  arr.sum() 耗时: 0.657 ms, 结果 = 499999500000
加速比: 19.8 倍


## 1.3 ndarray 的核心属性

创建一个数组后，最常用到的 6 个属性：

| 属性 | 含义 | 示例（shape=(2,3,4)） |
|---|---|---|
| `shape` | 各维度长度（元组） | `(2, 3, 4)` |
| `ndim` | 维数（shape 的长度） | `3` |
| `size` | 元素总数（各维度乘积） | `24` |
| `dtype` | 元素数据类型 | `int64` |
| `itemsize` | 每个元素占用的字节数 | `8` |
| `nbytes` | 总字节数（size × itemsize） | `192` |

> ⚠️ **陷阱**：`len(arr)` 只返回**第 0 维的长度**，不是元素总数！想知道总个数要用 `arr.size`。

In [3]:
# 一个三维数组：2 页 × 3 行 × 4 列
arr = np.arange(2 * 3 * 4).reshape(2, 3, 4)

print('数组内容：')
print(arr)
print('shape    =', arr.shape)     # 每个维度的长度
print('ndim     =', arr.ndim)      # 维数
print('size     =', arr.size)      # 元素总数
print('dtype    =', arr.dtype)     # 元素类型
print('itemsize =', arr.itemsize, '字节/元素')
print('nbytes   =', arr.nbytes, '字节 =', arr.size * arr.itemsize)
print('len(arr) =', len(arr))      # 注意：len 只返回第 0 维长度 2！

数组内容：
[[[ 0  1  2  3]
  [ 4  5  6  7]
  [ 8  9 10 11]]

 [[12 13 14 15]
  [16 17 18 19]
  [20 21 22 23]]]
shape    = (2, 3, 4)
ndim     = 3
size     = 24
dtype    = int64
itemsize = 8 字节/元素
nbytes   = 192 字节 = 192
len(arr) = 2


## 1.4 创建数组全览

NumPy 提供了非常多的创建方式，先看一张「决策图」帮你记忆：

```mermaid
flowchart TD
    A[要创建数组] --> B{数据从哪来}
    B -->|已有 list 或 tuple| C[np.array 或 np.asarray]
    B -->|等差数列| D{关心步长还是点数}
    D -->|步长| E[np.arange]
    D -->|点数| F[np.linspace]
    B -->|全 0 全 1 指定值| G[np.zeros 或 np.ones 或 np.full]
    B -->|单位矩阵 对角矩阵| H[np.eye 或 np.identity 或 np.diag]
    B -->|照着已有数组形状| I[np.zeros_like 或 np.ones_like]
```

> 💡 **提示**：创建数组最常用的两个参数是**形状 shape（元组）**和 **dtype**，如 `np.zeros((2, 3), dtype=float)`。

### 1.4.1 np.array：从 Python 容器创建（最常用）

`np.array()` 接收 list / tuple / 嵌套 list，自动推断 dtype：

- 一维 list → 一维数组
- 嵌套 list（形状规整）→ 多维数组
- 可以显式传入 `dtype=` 强制指定类型

In [4]:
# 从 list 创建
v1 = np.array([1, 2, 3])
print('从 list 创建  :', v1, ', dtype =', v1.dtype)

# 从嵌套 list 创建二维数组
m = np.array([[1, 2], [3, 4]])
print('二维数组：')
print(m)
print('shape =', m.shape)

# 从 tuple 创建
t = np.array((5, 6, 7))
print('从 tuple 创建 :', t)

# 创建时指定 dtype
f = np.array([1, 2, 3], dtype=np.float64)
print('指定 float64  :', f, ', dtype =', f.dtype)

从 list 创建  : [1 2 3] , dtype = int64
二维数组：
[[1 2]
 [3 4]]
shape = (2, 2)
从 tuple 创建 : [5 6 7]
指定 float64  : [1. 2. 3.] , dtype = float64


### 1.4.2 np.arange vs range vs np.linspace

| 函数 | 参数风格 | 特点 | 典型场景 |
|---|---|---|---|
| `range(a, b, s)` | 步长 s | Python 内置，返回 range 对象，只支持整数 | 纯 Python 循环 |
| `np.arange(a, b, s)` | 步长 s | 返回 ndarray，支持浮点步长 | 等差数列（步长已知） |
| `np.linspace(a, b, n)` | 点数 n | **固定点数**，自动算步长，默认包含端点 | 绘图坐标、采样点 |

> ⚠️ **陷阱**：浮点步长会让 `np.arange` 的结果「看起来不精确」。例如 `np.arange(0, 1, 0.2)` 里会出现 `0.6000000000000001`，因为 0.1/0.2 在二进制里无法精确表示。
>
> 💡 **提示**：当你「知道要多少个点」而不是「知道步长」时，一律用 `np.linspace`，它能保证端点是精确的。

In [5]:
# arange：步长式，左闭右开（不含终点）
print('np.arange(0, 10, 2)  =', np.arange(0, 10, 2))

# 浮点步长：结果出现 0.6000000000000001 这种“脏数字”
print('np.arange(0, 1, 0.2) =', np.arange(0, 1, 0.2))

# linspace：点数式，默认包含端点
print('np.linspace(0, 1, 11) =', np.linspace(0, 1, 11))

# endpoint=False 时不含终点
print('np.linspace(0, 1, 5, endpoint=False) =', np.linspace(0, 1, 5, endpoint=False))

np.arange(0, 10, 2)  = [0 2 4 6 8]
np.arange(0, 1, 0.2) = [0.  0.2 0.4 0.6 0.8]
np.linspace(0, 1, 11) = [0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1. ]
np.linspace(0, 1, 5, endpoint=False) = [0.  0.2 0.4 0.6 0.8]


### 1.4.3 固定值数组：zeros / ones / full / empty

| 函数 | 作用 |
|---|---|
| `np.zeros(shape)` | 全 0 数组 |
| `np.ones(shape)` | 全 1 数组 |
| `np.full(shape, value)` | 全部填成指定值 |
| `np.empty(shape)` | 只分配内存，**不初始化** |

> ⚠️ **陷阱**：`np.empty()` 返回的是「内存垃圾值」（之前残留的数据），数值不可预期！它只在你**马上会用赋值填满**时才有用（追求那一点点分配速度）。初学者看到奇怪的大数不要慌，那是垃圾值不是 bug。

In [6]:
print('np.zeros((2, 3)) ：')
print(np.zeros((2, 3)))

print('np.ones((2, 2), dtype=int) ：')
print(np.ones((2, 2), dtype=int))

print('np.full((2, 3), 7) ：')
print(np.full((2, 3), 7))

# empty 的“未初始化”陷阱：打印出来是垃圾值（不同机器/不同时刻结果不同）
e = np.empty((2, 3))
print('np.empty((2, 3)) ：（内容是“垃圾值”，不可预期）')
print(e)

np.zeros((2, 3)) ：
[[0. 0. 0.]
 [0. 0. 0.]]
np.ones((2, 2), dtype=int) ：
[[1 1]
 [1 1]]
np.full((2, 3), 7) ：
[[7 7 7]
 [7 7 7]]
np.empty((2, 3)) ：（内容是“垃圾值”，不可预期）
[[3.5e-323 3.5e-323 3.5e-323]
 [3.5e-323 3.5e-323 3.5e-323]]


### 1.4.4 形状复制：zeros_like / ones_like

`np.zeros_like(a)` / `np.ones_like(a)`：生成与 `a` **形状相同**（且默认 dtype 相同）的数组，非常适合「照着现有数组造骨架」。

### 1.4.5 特殊矩阵：eye / identity / diag

| 函数 | 作用 |
|---|---|
| `np.eye(n, k=0)` | n×n 单位矩阵；`k` 可偏移对角线 |
| `np.identity(n)` | 等价于 `eye(n)` |
| `np.diag(v)` | 传向量 → 构造对角矩阵；传矩阵 → 取出对角线 |

In [7]:
x = np.arange(6).reshape(2, 3)
print('x ：')
print(x)
print('np.zeros_like(x) ：')
print(np.zeros_like(x))
print('np.ones_like(x)  ：')
print(np.ones_like(x))

print('np.eye(3) ：')
print(np.eye(3))            # 单位矩阵
print('np.identity(3) ：')
print(np.identity(3))       # 等价于 eye(3)
print('np.eye(3, k=1) ：')
print(np.eye(3, k=1))       # 偏移对角线

m = np.arange(9).reshape(3, 3)
print('m ：')
print(m)
print('np.diag(m) =', np.diag(m))                  # 取矩阵对角线
print('np.diag([1,2,3]) ：')
print(np.diag([1, 2, 3]))                          # 由向量构造对角矩阵

x ：
[[0 1 2]
 [3 4 5]]
np.zeros_like(x) ：
[[0 0 0]
 [0 0 0]]
np.ones_like(x)  ：
[[1 1 1]
 [1 1 1]]
np.eye(3) ：
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
np.identity(3) ：
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
np.eye(3, k=1) ：
[[0. 1. 0.]
 [0. 0. 1.]
 [0. 0. 0.]]
m ：
[[0 1 2]
 [3 4 5]
 [6 7 8]]
np.diag(m) = [0 4 8]
np.diag([1,2,3]) ：
[[1 0 0]
 [0 2 0]
 [0 0 3]]


## 1.5 dtype：元素的数据类型

`dtype` 是 ndarray 的灵魂——它决定了每个元素占几个字节、怎么解释二进制位。这也是「同构」的体现。

### 常用类型一览

| dtype | 说明 | 示例 |
|---|---|---|
| `np.int8 / int16 / int32 / int64` | 有符号整数 | `np.array([1,2], dtype=np.int32)` |
| `np.uint8 ... uint64` | 无符号整数 | — |
| `np.float16 / float32 / float64` | 浮点数 | 默认浮点就是 float64 |
| `np.bool_` | 布尔 | `np.array([True, False])` |
| `np.str_` | 字符串（定长） | `np.array(['a','b'])` |
| `np.object_` | Python 对象（退化为慢速） | 尽量避免 |

类型层次与转换：

```mermaid
mindmap
  root(dtype 类型体系)
    bool 布尔
    int 整数
      int8
      int16
      int32
      int64
    uint 无符号
    float 浮点
      float32
      float64
    complex 复数
    str 字符串
    object Python 对象
```

> ⚠️ **提示**：Python 内置的 `int` 和 NumPy 的 `np.int64` 不是一回事；Python 的 int 是「变长」的，NumPy 的整数是「定长」的，定长就会有**溢出**问题（见 1.5.3）。

In [8]:
# 查看各类型的 dtype 名称
for t in (np.int32, np.int64, np.float64, np.bool_, np.object_):
    print(f'{str(t):>14} -> {np.dtype(t)}')

# 字符串数组是【定长】Unicode 类型（str_），长度取最长元素
print('字符串数组的 dtype =', np.array(['a', 'bc']).dtype)   # <U2

# 创建时指定 dtype
a = np.array([1, 2, 3], dtype=np.float32)
print('a =', a, ', dtype =', a.dtype)

# astype 转换：返回【新数组】（拷贝），不修改原数组
b = a.astype(np.int64)
print('b =', b, ', dtype =', b.dtype)
print('转换后原数组 a 的 dtype 仍是:', a.dtype)
print('astype 结果是否与原数组共享内存:', np.shares_memory(a, b))  # False：是拷贝

<class 'numpy.int32'> -> int32
<class 'numpy.int64'> -> int64
<class 'numpy.float64'> -> float64
<class 'numpy.bool'> -> bool
<class 'numpy.object_'> -> object
字符串数组的 dtype = <U2
a = [1. 2. 3.] , dtype = float32
b = [1 2 3] , dtype = int64
转换后原数组 a 的 dtype 仍是: float32
astype 结果是否与原数组共享内存: False


### 1.5.2 类型提升规则

不同 dtype 的数组做运算时，NumPy 会自动把「级别低」的类型提升到「级别高」的：

```
bool < int < uint < float < complex
```

- `int + float → float`
- `int32 + int64 → int64`（精度更高者胜）
- `bool + int → int`

可以用 `np.result_type(t1, t2)` 来**预测**两个类型运算后的结果类型（不会真的做运算）。

In [9]:
# 类型提升：int + float → float
i = np.array([1, 2, 3], dtype=np.int64)
f = np.array([1.5, 2.5, 3.5], dtype=np.float64)
print('int64 + float64 的结果 dtype =', (i + f).dtype)

# bool + int → int
b = np.array([True, False])
print('bool + int64    的结果 dtype =', (b + i[:2]).dtype)

# 用 np.result_type 预测结果类型
print('result_type(int64, float64)  =', np.result_type(np.int64, np.float64))
print('result_type(int8, int64)     =', np.result_type(np.int8, np.int64))
print('result_type(float32, int64)  =', np.result_type(np.float32, np.int64))

int64 + float64 的结果 dtype = float64
bool + int64    的结果 dtype = int64
result_type(int64, float64)  = float64
result_type(int8, int64)     = int64
result_type(float32, int64)  = float64


### 1.5.3 整数溢出陷阱

定长整数有取值范围。比如 `int8` 只能表示 -128 ~ 127。一旦结果超出范围，就会**回绕（wrap around）**：

> ⚠️ **陷阱**：溢出不会报错终止程序，而是静默得到错误结果！`127 + 1` 在 int8 下会变成 `-128`（早期 NumPy 版本会发 `RuntimeWarning`，NumPy 2.x 起静默回绕）。涉及可能溢出的运算时，要么改用更大位宽（如 int64），要么改用浮点。
>
> 另外，NumPy 2.x 起，把**超出范围的值直接 cast 成小类型**会直接抛 `OverflowError`，反而更安全——能尽早暴露问题而不是静默出错。

In [10]:
# 陷阱 1：int8 加法溢出 → 回绕成 -128
x = np.array([127], dtype=np.int8)
y = np.array([1], dtype=np.int8)
print('127 + 1 (int8) =', x + y)  # 回绕成 -128

# 陷阱 2：int64 一样会溢出（最大值再加 1）
big = np.array([np.iinfo(np.int64).max], dtype=np.int64)
print('int64 最大值 + 1 =', big + 1)

# 陷阱 3：NumPy 2.x 中，把超出范围的值 cast 成小类型会直接报错
print('cast 200 -> int8 会报错：')
try:
    np.array([200], dtype=np.int8)
except OverflowError as e:
    print('  OverflowError:', e)

# 观察各类型范围
print('int8  范围 =', np.iinfo(np.int8).min, '~', np.iinfo(np.int8).max)
print('int64 范围 =', np.iinfo(np.int64).min, '~', np.iinfo(np.int64).max)
print('float64 信息 =', np.finfo(np.float64))

127 + 1 (int8) = [-128]
int64 最大值 + 1 = [-9223372036854775808]
cast 200 -> int8 会报错：
  OverflowError: Python integer 200 out of bounds for int8
int8  范围 = -128 ~ 127
int64 范围 = -9223372036854775808 ~ 9223372036854775807
float64 信息 = Machine parameters for float64
---------------------------------------------------------------
precision = 15   resolution = 1e-15
machep = -52   eps =        2.220446049250313e-16
negep =  -53   epsneg =     1.1102230246251565e-16
minexp = -1022   tiny =       2.2250738585072014e-308
maxexp = 1024   max =        1.7976931348623157e+308
nexp =   11   min =        -max
smallest_normal = 2.2250738585072014e-308   smallest_subnormal = 5e-324
---------------------------------------------------------------



## 1.6 数组与 Python 容器的互转

### 数组 → 列表：`arr.tolist()`

`tolist()` 把 ndarray 转回嵌套的 Python list，常用于把结果交给其他纯 Python 库。

### `np.asarray` vs `np.array`（重点对比）

| 函数 | 输入是 ndarray | 输入是 list / tuple | 复制行为 |
|---|---|---|---|
| `np.array(a)` | 生成一份**拷贝** | 转成新的 ndarray | 总是新建 |
| `np.asarray(a)` | **原样返回**（不复制） | 转成新的 ndarray | 已有数组不复制 |

> 💡 **提示**：当你「不确定传入的已经是数组、又不想白白复制一份」时，用 `np.asarray`。写库函数时用它接收输入很常见。

In [11]:
# 数组 -> Python 列表
a = np.arange(5)
print('a.tolist() =', a.tolist(), ', 类型 =', type(a.tolist()))

m = np.arange(4).reshape(2, 2)
print('二维 tolist() =', m.tolist())

# asarray 与 array 的区别：asarray 不复制已有的 ndarray
b = np.arange(5)
c1 = np.asarray(b)   # 不复制
c2 = np.array(b)     # 复制
print('np.asarray 是否共享内存:', np.shares_memory(b, c1))  # True
print('np.array   是否共享内存:', np.shares_memory(b, c2))  # False

# asarray 对 Python list 也会转成 ndarray
print('np.asarray([1, 2, 3]) =', np.asarray([1, 2, 3]), ', dtype =', np.asarray([1, 2, 3]).dtype)

a.tolist() = [0, 1, 2, 3, 4] , 类型 = <class 'list'>
二维 tolist() = [[0, 1], [2, 3]]
np.asarray 是否共享内存: True
np.array   是否共享内存: False
np.asarray([1, 2, 3]) = [1 2 3] , dtype = int64


## 1.7 本章小结

### 一句话记忆表

| 主题 | 一句话记住 |
|---|---|
| 为什么快 | 连续内存 + C 实现 + 向量化 |
| 核心属性 | `shape` 形状、`size` 总数、`dtype` 类型、`nbytes` 总字节 |
| 创建数组 | 已有数据用 `np.array`；等差看步长 `arange` / 看点数 `linspace`；固定值 `zeros/ones/full`；`empty` 是垃圾值 |
| dtype 转换 | `astype` 返回**拷贝**；`int + float → float` |
| 溢出 | 定长整数会回绕，运算时留意范围 |
| 互转 | 数组 → 列表用 `tolist()`；`asarray` 不复制已有数组 |

本章知识地图：

```mermaid
flowchart LR
    A[ndarray 基础] --> B[为什么快]
    A --> C[核心属性]
    A --> D[创建数组]
    A --> E[dtype]
    A --> F[与 Python 容器互转]
    D --> G[array arange linspace]
    D --> H[zeros ones full empty]
    D --> I[eye identity diag]
    E --> J[astype 转换]
    E --> K[类型提升]
    E --> L[溢出陷阱]
```

### 📝 动手练习（建议先自己写，再看提示）

1. **创建**：用一行代码创建一个 5×5 的数组，主对角线上是 1，其余全是 0。
2. **属性**：用 `np.arange(24).reshape(2, 3, 4)` 创建数组，打印它的 `shape`、`ndim`、`size`、`itemsize`、`nbytes`，并验证 `len()` 只等于第 0 维长度。
3. **转换与精度**：把 `np.linspace(0, 1, 5)` 转成列表，再用 `astype(np.float32)` 转成单精度，对比两次结果的差异；最后用 `np.result_type` 预测 `np.float32` 与 `np.int64` 运算的结果类型。

> 💡 **练习提示**
>
> - 第 1 题：`np.eye(5)` 直接搞定；或 `np.diag(np.ones(5))`。
> - 第 2 题：`size == 2*3*4 == 24`，`nbytes == size * itemsize`。
> - 第 3 题：float64 → float32 会把 0.1 存成更粗糙的值，`print` 出来能看到尾部多出的「脏数字」；`np.result_type(np.float32, np.int64)` 结果是 `float64`。

👉 **下一章**：`02_索引切片与布尔筛选.ipynb` —— 学会如何「精准取数」，包括索引、切片、视图、布尔筛选与 `np.where`。